# AIPV1EMI0001 — build and deploy an AI Processor on an EdgeAI Station

This notebook rebuilds **AIPV1EMI0001** (YOLOv7-tiny detector for `car`, `person`, `truck`)
from source, packages it with **`eais-tools`**, runs it in a real station pipeline and installs
its Node-RED dashboard. Each cell is one visible command (or a short check); run them top to bottom.

## What is an AI Processor?

An **AI Processor (AIP)** is a self-describing folder that the station turns into a GPU video
pipeline. The manifest declares the model, tracker and analytics; the station builds the
DeepStream pipeline from it; `callback.py` turns per-frame metadata into business data that
Node-RED and the dashboard consume.

```mermaid
flowchart LR
    subgraph AIP["AI Processor folder: /data/aip/AIPV1EMI0001"]
        M["manifest.json<br/>identity, classes, batch,<br/>tracker, analytics"]
        C["configs/<br/>nvinferserver + NvDCF tracker"]
        MO["models/AIPV1EMI0001_Yolo_Detector/<br/>config.pbtxt, labels,<br/>parser .so, 1/*.engine"]
        CB["callback.py<br/>metadata to JSON"]
        L["logic/<br/>Node-RED flow"]
        D["dashboard/<br/>UIBuilder app"]
    end
    HF[("Hugging Face<br/>ONNX + prebuilt engine")] --> MO
    AIP -- "eais-tools aip build / export" --> PKG[["AIPV1EMI0001_dist.tgz"]]
    PKG --> ST["Any compatible<br/>EdgeAI Station"]
```

## What runs on the station

```mermaid
flowchart LR
    SRC["Cameras / files<br/>up to 8 streams"] --> DEC["Hardware decode<br/>nvurisrcbin"]
    DEC --> MUX["Batching<br/>nvstreammux (batch 8)"]
    MUX --> INF["Inference<br/>nvinferserver + TensorRT FP16<br/>custom NMS parser"]
    INF --> TRK["Tracking<br/>nvtracker NvDCF"]
    TRK --> ANA["Analytics<br/>ROI, line crossing"]
    ANA --> CB["callback.py<br/>counts per second"]
    CB -- "callback messages" --> NR["Node-RED<br/>logic/"]
    NR --> UI["Dashboard<br/>counts + HLS video"]
```

Frames stay in GPU memory from decode to analytics; only metadata reaches Python.
The inference server runs **inside** the DeepStream process (`nvinferserver`), with no
network hop to a separate model server.

## Steps in this notebook

```mermaid
flowchart TD
    S0["0. Setup"] --> S1["1. Download weights<br/>Hugging Face + SHA-256"]
    S1 --> S2["2. Build<br/>make (parser), trtexec (engine)"]
    S2 --> S3{"3. Benchmark<br/>(optional)"}
    S3 --> S4["4. Validate<br/>eais-tools aip check + config validators"]
    S4 --> S5["5. Package<br/>eais-tools aip build / export"]
    S5 --> S6["6. Scan<br/>eais-tools aip scan"]
    S6 --> S7["7. Pipeline test<br/>eais-tools pipeline test"]
    S7 --> S8["8. Node-RED + dashboard"]
    S8 --> S9["9. Cleanup"]
```

Mermaid diagrams render in JupyterLab 4.1+ and in the repository README. For the background
and design choices behind this AIP, see [`presentation.ipynb`](presentation.ipynb).

## 0. Setup

Run this notebook in the station's Jupyter environment (**App-Builder**, `ms8_jupyter`):
it provides `eais-tools`, CUDA, DeepStream headers and `trtexec`, and shares `/data/aip`
(AI processors) and `/data/projects` (Node-RED projects) with the station services.

The station only discovers AI processors under `/data/aip/<unique_id>`. From a Jupyter terminal:

```bash
cd /data/aip && git clone https://github.com/edge-ai/AIPV1EMI0001.git
```

then open `/data/aip/AIPV1EMI0001/localdev.ipynb`. Move an earlier install aside first
(`mv AIPV1EMI0001 AIPV1EMI0001.backup.$(date +%s)`).

Tested with JetPack 6.0 / L4T R36.3 / TensorRT 8.6.2 / DeepStream 7.0 / `eais-tools` 0.9.2.

Choose the options below. The `%%bash` cells read them as environment variables and stop
at the first failing command (`set -euo pipefail`).

| Option | Meaning |
|---|---|
| `ENGINE_SOURCE = "build"` | Download the ONNX and compile the engine on this station (30–60 min on a busy Orin NX) |
| `ENGINE_SOURCE = "prebuilt"` | Download the published FP16 engine; only loads on **TensorRT 8.6.2** (JetPack 6 / L4T R36.3) |
| `RUN_BENCHMARKS = True` | Also run section 3: two extra engines and a fair ONNX Runtime / TensorRT comparison (30–60+ min) |
| `WORKSPACE_MIB` | TensorRT builder workspace limit, see section 2 |

In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

AIP_ID = "AIPV1EMI0001"
ENGINE_SOURCE = "build"  # "build" or "prebuilt"
RUN_BENCHMARKS = False
WORKSPACE_MIB = 1024

assert ENGINE_SOURCE in ("build", "prebuilt"), ENGINE_SOURCE
assert Path.cwd() == Path("/data/aip") / AIP_ID, f"Open this notebook from /data/aip/{AIP_ID}"
MODEL_DIR = f"models/{AIP_ID}_Yolo_Detector"
os.environ.update(
    AIP_ID=AIP_ID, ENGINE_SOURCE=ENGINE_SOURCE, RUN_BENCHMARKS=str(int(RUN_BENCHMARKS)),
    WORKSPACE_MIB=str(WORKSPACE_MIB), MODEL_DIR=MODEL_DIR,
    ONNX=f"{MODEL_DIR}/yolov7tiny-640.onnx",
    ENGINE=f"{MODEL_DIR}/1/yolov7-fp16-1x8x8.engine",
    TRTEXEC=os.environ.get("TRTEXEC_BIN", "/usr/src/tensorrt/bin/trtexec"),
    HF_BASE="https://huggingface.co/Edgematrix-JP/AIPV1EMI0001/resolve/main",
    MS3="http://ms3_videostreaming:8000",
    BENCH="tmp/inference-benchmark/" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
)
print(f"ENGINE_SOURCE={ENGINE_SOURCE} RUN_BENCHMARKS={RUN_BENCHMARKS} WORKSPACE_MIB={WORKSPACE_MIB}")

In [ ]:
%%bash
set -euo pipefail
for tool in eais-tools make git npm jq curl timeout "$TRTEXEC"; do
    command -v "$tool" > /dev/null || { echo "missing: $tool"; exit 1; }
done
test -d /opt/nvidia/deepstream/deepstream/sources/includes
head -n 1 /etc/nv_tegra_release
/usr/local/cuda/bin/nvcc --version | grep release
banner=$("$TRTEXEC" --help 2>&1 || true)
[[ $banner =~ TensorRT\ v[0-9]+ ]] && echo "${BASH_REMATCH[0]}"
df -h /data/aip | tail -n 1

## 1. Download the weights

The weights are **not stored in Git**. They are published on Hugging Face at
[`Edgematrix-JP/AIPV1EMI0001`](https://huggingface.co/Edgematrix-JP/AIPV1EMI0001)
(GPL-3.0, like the YOLOv7 training code) and verified here with SHA-256.

- `yolov7tiny-640.onnx` — portable source model, with NMS (`EfficientNMS_TRT`) in the graph.
- `yolov7-fp16-1x8x8-jp6-l4t36.3-arm64.engine` — compiled TensorRT engine. An engine is tied
  to the exact TensorRT version and GPU it was built for, so the cell refuses it elsewhere.

In [ ]:
%%bash
set -euo pipefail
fetch() {  # fetch <file on Hugging Face> <local path> <sha256>
    if ! echo "$3  $2" | sha256sum --check --status 2> /dev/null; then
        mkdir -p "$(dirname "$2")"
        curl -fL --retry 3 -o "$2.part" "$HF_BASE/$1"
        echo "$3  $2.part" | sha256sum --check --quiet
        mv "$2.part" "$2"
    fi
    echo "OK  $2"
}

if [ "$ENGINE_SOURCE" = build ] || [ "$RUN_BENCHMARKS" = 1 ]; then
    fetch yolov7tiny-640.onnx "$ONNX" \
        5796e9e0836751e3b7a9623cc6db976ac7fd087fcfb2742d20d06dcf0a0b3560
fi
if [ "$ENGINE_SOURCE" = prebuilt ]; then
    banner=$("$TRTEXEC" --help 2>&1 || true)
    [[ $banner == *"TensorRT v8602"* ]] || {
        echo "The prebuilt engine needs TensorRT 8.6.2: use ENGINE_SOURCE = \"build\""; exit 1; }
    fetch yolov7-fp16-1x8x8-jp6-l4t36.3-arm64.engine "$ENGINE" \
        670150b495a0f120d178fbbb1e3f29758ccbee710e4477bf6847f46517a873b8
fi

## 2. Build the runtime artifacts

`bin/build.sh` performs the same two builds non-interactively; do not run both.

### 2.1 Custom parser

`make` compiles `src/nvdsinfer_customparser/` against this station's CUDA and DeepStream
headers. The library converts the model's NMS outputs into DeepStream detections.

In [ ]:
%%bash
set -euo pipefail
CUDA_VER=$(/usr/local/cuda/bin/nvcc --version | sed -n 's/.*release \([0-9.]*\),.*/\1/p')
make -C src/nvdsinfer_customparser CUDA_VER="$CUDA_VER"
cp src/nvdsinfer_customparser/libnvds_infercustomparser.so "$MODEL_DIR/"
ls -l "$MODEL_DIR/libnvds_infercustomparser.so"

### 2.2 TensorRT engine

`trtexec` compiles the ONNX graph into an engine optimized for **this** GPU and TensorRT version.

| Flag | Meaning |
|---|---|
| `--fp16` | Allow FP16 kernels (with higher-precision fallbacks): typically several times faster on Jetson |
| `--minShapes` / `--optShapes` / `--maxShapes` | Dynamic batch profile 1 / 8 / 8: one engine serves batches 1 to 8, tuned for 8 (= `max_batch_size` in the configs) |
| `--workspace` | Builder scratch memory limit in MiB (TensorRT 8.6 syntax). Jetson shares RAM between CPU and GPU: 4096 MiB was OOM-killed with the station stack running, 1024 is safe |
| `--timingCacheFile` | Reuse kernel timings to speed up later builds (not an engine cache) |
| `--skipInference` | Build only; the next cell runs a separate smoke test |

Changing the batch or resolution also requires changing `config.pbtxt`, the DeepStream config
and the manifest. The build log goes to `tmp/build/`; its tail is printed on failure.

In [ ]:
%%bash
set -euo pipefail
if [ "$ENGINE_SOURCE" = prebuilt ]; then echo "Using the prebuilt engine $ENGINE"; exit 0; fi
mkdir -p "$(dirname "$ENGINE")" tmp/build
"$TRTEXEC" --onnx="$ONNX" --fp16 --workspace="$WORKSPACE_MIB" \
    --minShapes=images:1x3x640x640 --optShapes=images:8x3x640x640 --maxShapes=images:8x3x640x640 \
    --saveEngine="$ENGINE" --timingCacheFile=bin/timing.cache --skipInference \
    > tmp/build/deployment-fp16.log 2>&1 || { tail -n 60 tmp/build/deployment-fp16.log; exit 1; }
ls -lh "$ENGINE"

In [ ]:
%%bash
set -euo pipefail
mkdir -p tmp/build
"$TRTEXEC" --loadEngine="$ENGINE" --shapes=images:1x3x640x640 --duration=1 --warmUp=200 --iterations=1 \
    > tmp/build/smoke-test.log 2>&1 || { tail -n 40 tmp/build/smoke-test.log; exit 1; }
grep -q '&&&& PASSED' tmp/build/smoke-test.log
echo "OK: the engine loads and runs on this station"

## 3. Optional: fair ONNX Runtime vs TensorRT benchmark

Skipped unless `RUN_BENCHMARKS = True`. It answers *“how much faster is the TensorRT engine than
running the ONNX model directly?”* without comparing apples and oranges:

- **Same graph**: a copy of the ONNX without the final NMS node (ONNX Runtime cannot run
  TensorRT's NMS plugin). The deployment model is not modified.
- **Same profile and flags**: both engines use batch 1/8/8, the same workspace and `--noTF32`;
  only `--fp16` differs. ONNX Runtime uses its CUDA provider with TF32 disabled.
- **Same timing boundary**, as in [Ultralytics' predictor](https://github.com/ultralytics/ultralytics/blob/80c34c050921abd29970d8e3d2a2a41ec6538c98/ultralytics/engine/predictor.py#L380-L397):
  `cudaDeviceSynchronize → perf_counter → one call → cudaDeviceSynchronize`, with shared
  GPU-resident input, preallocated GPU outputs, 20 warm-up + 50 timed requests × 3 rounds, in
  shuffled order.

`trtexec` only **builds** the engines: its own timings exclude Python dispatch and pipeline
requests, so they cannot share a table with ONNX Runtime. Both runtimes execute in one Python
process ([`tools/benchmark.py`](tools/benchmark.py)), without a model server. TensorRT's Python
bindings are extracted from NVIDIA's APT repository at the exact installed version, under `tmp/`
only (no system install). Results are raw model latency, not pipeline FPS; for pipeline-level
throughput use `eais-tools check perf`. Reference results are in the README.

In [ ]:
%%bash
set -euo pipefail
[ "$RUN_BENCHMARKS" = 1 ] || { echo "Skipped (RUN_BENCHMARKS = False)"; exit 0; }
mkdir -p "$BENCH"
python3 tools/benchmark.py raw-onnx "$ONNX" "$BENCH/raw.onnx"

In [ ]:
%%bash
set -euo pipefail
[ "$RUN_BENCHMARKS" = 1 ] || { echo "Skipped (RUN_BENCHMARKS = False)"; exit 0; }
build() {  # build <name> [precision flag]: identical profile and workspace, only precision differs
    local name=$1; shift
    "$TRTEXEC" --onnx="$BENCH/raw.onnx" --noTF32 "$@" --workspace="$WORKSPACE_MIB" \
        --minShapes=images:1x3x640x640 --optShapes=images:8x3x640x640 --maxShapes=images:8x3x640x640 \
        --saveEngine="$BENCH/$name.engine" --timingCacheFile="$BENCH/$name.cache" --skipInference \
        > "$BENCH/$name.log" 2>&1 || { tail -n 60 "$BENCH/$name.log"; return 1; }
    ls -lh "$BENCH/$name.engine"
}
build raw-fp32
build raw-fp16 --fp16

In [ ]:
%%bash
set -euo pipefail
[ "$RUN_BENCHMARKS" = 1 ] || { echo "Skipped (RUN_BENCHMARKS = False)"; exit 0; }
python3 tools/benchmark.py run --onnx "$BENCH/raw.onnx" \
    --engine "TensorRT FP32=$BENCH/raw-fp32.engine" --engine "TensorRT FP16=$BENCH/raw-fp16.engine" \
    --output "$BENCH" --source-onnx "$ONNX" --deployment-engine "$ENGINE" \
    --trtexec "$TRTEXEC" --workspace-mib "$WORKSPACE_MIB"

## 4. Validate

`eais-tools aip check` verifies the manifest, required files, model folders and the callback class.
The next cell runs the EAIS Triton and DeepStream config validators directly: the CLI wrappers
(`eais-tools configs triton check` / `configs ds check`) run the same validators but then start a
blocking Triton server or a `gst-launch` test. NvDCF (OpenCV YAML) gets a basic structure check.

Static checks do not run the tracker on the GPU: the pipeline test in section 7 does.

In [ ]:
%%bash
set -euo pipefail
eais-tools aip check --uuid "$AIP_ID"

In [ ]:
import json, yaml
from eais_tools.ds_config_validator import validate_ds_config
from eais_tools.services.configs_service import validate_model_config
from eais_tools.triton_config_validator import validate_specified_triton_config

manifest = json.loads(Path("manifest.json").read_text())
detector, tracker = manifest["primary_processor"], manifest["tracker_engine"]
pbtxt = str(Path(MODEL_DIR, "config.pbtxt").resolve())
triton = validate_specified_triton_config(pbtxt)
assert triton.result and validate_model_config(detector["name"], pbtxt), triton.error_message
ds = validate_ds_config(str(Path("configs", detector["config_plan"]).resolve()))
assert ds.success, ds.message
nvdcf = yaml.safe_load(Path("configs", tracker["config_plan"]).read_text().removeprefix("%YAML:1.0"))
assert {"BaseConfig", "TargetManagement", "StateEstimator", "VisualTracker"} <= nvdcf.keys()
assert Path(tracker["lib_path"]).is_file(), tracker["lib_path"]
print("OK: Triton, DeepStream and NvDCF configurations")

## 5. Package

`eais-tools aip build` copies the runtime files into `dist/`: manifest, callback, root Markdown
files, `resources/logo.png`, `configs/`, each model's config/labels/parser and `1/` folder.
**`--include-file .include`** adds what the station also needs: licenses, documentation, the
Node-RED flow and the dashboard. `.include` lists individual files, because EAIS Tools copies
directories recursively and does not honor `.gitignore` (it would ship `node_modules/` or
`flows_cred.json`).

[`tools/check_package.py`](tools/check_package.py) then proves the result: `dist/` must contain
exactly the expected files with the same bytes as the sources (no ONNX, credentials, caches or
symlinks), and the exported archive must match `dist/`. The CI release workflow runs the same check.

In [ ]:
%%bash
set -euo pipefail
eais-tools aip build --uuid "$AIP_ID" --include-file .include
python3 tools/check_package.py
find dist -type f | sort

In [ ]:
%%bash
set -euo pipefail
rm -f "${AIP_ID}_dist.tgz"
eais-tools aip export --uuid "$AIP_ID"
python3 tools/check_package.py --archive "${AIP_ID}_dist.tgz"
sha256sum "${AIP_ID}_dist.tgz"

## 6. Make the AIP available to the station

The folder is already in `/data/aip`, so instead of uploading the archive we ask the
video-streaming service (ms3) to scan it, and check that it found this version.
(`AIPV1EMI0001_dist.tgz` is what you install on *other* stations.)

No model-server registration is needed: `nvinferserver` runs the Triton library inside each
DeepStream pipeline. (The ms3 `register` endpoint only loads models into the separate ms7
Triton server, which this AIP does not use.)

In [ ]:
%%bash
set -euo pipefail
eais-tools aip scan
AIP=$(curl -fsS "$MS3/aiprocessors" | jq -ec --arg id "$AIP_ID" '.[] | select(.unique_id == $id)')
test "$(jq -r .version <<< "$AIP")" = "$(jq -r .version manifest.json)"
echo "OK: ms3 lists $AIP_ID v$(jq -r .version <<< "$AIP") (pk $(jq -r .pk <<< "$AIP"))"

## 7. Run a pipeline test

**`eais-tools pipeline test`** is the runtime acceptance test: it uploads a sample video and
runs a station-managed pipeline with this AIP's inference, NvDCF tracker and callback
(headless), recording callback messages to `tmp/callback_dump.json`. It listens until
interrupted, so `timeout` stops it with `SIGINT` after 90 s (exit code 124 is expected).
`--retain_resources` keeps the pipeline running for the dashboard in section 8.

The tool replaces any existing `eais_*` test resources, so the cell first refuses to run if
another test owns some.

In [ ]:
%%bash
set -euo pipefail
eais_resources() { curl -fsS "$MS3/$1" | jq '[.[] | select(.name | startswith("eais_"))] | length'; }
[ "$(eais_resources pipelines)" = 0 ] && [ "$(eais_resources medias)" = 0 ] || {
    echo "eais_* pipelines or media already exist: another test owns them"; exit 1; }
mkdir -p tmp && rm -f tmp/callback_dump.json
status=0
COLUMNS=160 timeout --signal=INT --kill-after=30 90 eais-tools pipeline test --aip "$AIP_ID" \
    --inputs_num 1 --callback_output --retain_resources --dump_file tmp/callback_dump.json || status=$?
[ "$status" = 0 ] || [ "$status" = 124 ]

A started pipeline is not a pass: the next cell requires **detected objects** in callback messages
that follow the [callback contract](doc/USER_MANUAL.md#callback-output-metadata-contract) —
`{since, until, data}` records where `data` maps each source to per-class counts and analytics.

In [ ]:
import json, requests

text, pos, records, decoder = Path("tmp/callback_dump.json").read_text(), 0, [], json.JSONDecoder()
while pos < len(text):  # the dump is a sequence of JSON arrays, one per message
    if text[pos].isspace():
        pos += 1
        continue
    batch, pos = decoder.raw_decode(text, pos)
    records.extend(batch)
assert records, "no callback messages received"

peak = {c: 0 for c in manifest["primary_processor"]["classes"]}
for record in records:
    assert {"since", "until", "data"} <= record.keys(), record.keys()
    for source in record["data"].values():
        assert source["obj_counter"].keys() == peak.keys(), source["obj_counter"].keys()
        assert {"objInROIcnt", "objLCCurrCnt"} <= source["analytics_obj_cnt"].keys()
        for c in peak:
            peak[c] = max(peak[c], source["obj_counter"][c]["max_val"])
print(f"{len(records)} callback records, peak objects per frame: {peak}")
assert any(peak.values()), "no objects detected"

pipelines = requests.get(f"{os.environ['MS3']}/pipelines", timeout=30).json()
(PIPELINE_ID,) = [p["pk"] for p in pipelines if p["name"] == "eais_pipeline"]
os.environ["PIPELINE_ID"] = PIPELINE_ID
print(f"OK: test pipeline {PIPELINE_ID} delivers detections and valid metadata")

## 8. Install the Node-RED flow and dashboard

The AIP ships a Node-RED project (`logic/`) and its UIBuilder dashboard (`dashboard/`).
The cell installs them as the Node-RED project **`nodered-aipv1emi0001-dashboard`**, wired to the
test pipeline through the `AIP_PIPELINE_ID` variable of the **AIPV1EMI0001 Logic** tab.
`npm ci` vendors the browser libraries (Vue, hls.js), so the dashboard then works offline;
this step itself needs npm registry access. `flows_cred.json` is never copied.
Set `OVERWRITE=1` to replace an earlier install. The active Node-RED project is not switched.

In [ ]:
%%bash
set -euo pipefail
NR_PROJECT=nodered-aipv1emi0001-dashboard
OVERWRITE=0
P=/data/projects/$NR_PROJECT
TAB='.[] | select(.type == "tab" and .label == "AIPV1EMI0001 Logic")'
test "$(jq "[$TAB] | length" logic/flows.json)" = 1
if [ -e "$P" ]; then
    [ "$OVERWRITE" = 1 ] || { echo "$P exists: set OVERWRITE=1 to replace it"; exit 1; }
    rm -rf "$P"
fi
mkdir -p "$P/uibuilder"
cp logic/package.json logic/package-lock.json logic/README.md "$P/"
jq --indent 4 --arg id "$PIPELINE_ID" \
    "($TAB).env = [{name: \"AIP_PIPELINE_ID\", value: \$id, type: \"str\"}]" \
    logic/flows.json > "$P/flows.json"
tar -C dashboard --exclude=node_modules --exclude=README.md -cf - . | tar -C "$P/uibuilder" -xf -
printf 'node_modules/\nflows_cred.json\n.flows*.backup\n' > "$P/.gitignore"
(cd "$P/uibuilder" && npm ci --omit=dev --no-audit --no-fund)
test -f "$P/uibuilder/node_modules/vue/package.json"
test -f "$P/uibuilder/node_modules/hls.js/package.json"
git -C "$P" init -q -b main
git -C "$P" add -A
git -C "$P" -c user.name="EdgeAI Station" -c user.email=edgeai@localhost commit -q -m "Install $AIP_ID dashboard"
echo "OK: Node-RED project $P wired to pipeline $PIPELINE_ID"

### Open the dashboard

These steps are manual on purpose: opening a project replaces the flows running in Node-RED
for every user of the station.

1. Open the Node-RED editor from the station web UI.
2. Menu (☰) → **Projects** → **Open** → `nodered-aipv1emi0001-dashboard`.
3. Open `https://<station>/ms5/aipv1emi0001_dash/`: `car`, `person` and `truck` counts update
   every second, next to the pipeline's HLS video.
4. To share it, assign the `aipv1emi0001_dash` dashboard to users in **Accounts management**.

To follow another pipeline, change `AIP_PIPELINE_ID` in the **AIPV1EMI0001 Logic** tab and deploy.

## 9. Cleanup

`eais-tools pipeline clean` deletes **all** `eais_*` pipelines and media, so the cell first checks
that they are only this run's. The dashboard stops receiving data until you point
`AIP_PIPELINE_ID` at another pipeline.

Kept: `AIPV1EMI0001_dist.tgz` and `dist/` (download the archive to install on another compatible
station), the engine and parser (ms3 copies the AIP folder each time a pipeline starts) and
benchmark results under `tmp/inference-benchmark/`. Removed: the downloaded ONNX and timing cache.

In [ ]:
%%bash
set -euo pipefail
curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" \
    '[.[] | select(.name | startswith("eais_")) | .pk] == [$id]' > /dev/null
curl -fsS "$MS3/medias" | jq -e '[.[] | select(.name | startswith("eais_"))] | length <= 1' > /dev/null
eais-tools pipeline clean
curl -fsS "$MS3/pipelines" | jq -e --arg id "$PIPELINE_ID" 'all(.[]; .pk != $id)' > /dev/null
rm -f "$ONNX" bin/timing.cache
sha256sum "${AIP_ID}_dist.tgz"